ASSIGNMENT NLP – 5 (Token Classification: POS Tagging & Chunking)

Assignment: Fine-Tuning BERT for POS Tagging & Chunking


In [3]:
!pip install transformers datasets evaluate seqeval accelerate -U

In [4]:
!pip install "datasets<3.0.0" -U

In [5]:
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification, pipeline

# ==========================================
# TASK 1: Dataset Selection

print("--- TASK 1: Dataset Selection ---")
# Loading CoNLL-2003 for Chunking (trust_remote_code required for the older datasets library version)
dataset = load_dataset("conll2003", trust_remote_code=True)

# Extracting the label list for chunking
label_list = dataset["train"].features["chunk_tags"].feature.names
print(f"Dataset selected: CoNLL-2003")
print(f"Label categories (Chunking): {label_list}\n")

# ==========================================
# TASK 2: Data Preprocessing

print("--- TASK 2: Data Preprocessing ---")
model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint) # Tokenize using BERT tokenizer

def tokenize_and_align_labels(examples):
    """
    Tokenizes inputs and aligns the chunk tags with the subwords created by the tokenizer.
    Handles subwords and assigns the special -100 token where needed.
    """
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)

    labels = []
    for i, label in enumerate(examples["chunk_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            # Special tokens mapped to -100
            if word_idx is None:
                label_ids.append(-100)
            # Set label for the first token of each word
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            # For subwords, set to -100 so they are ignored in loss calculation
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)
print("Preprocessing complete. Output includes input_ids, attention_mask, and labels.\n")

# ==========================================
# TASK 3: Model Setup 

print("--- TASK 3: Model Setup ---")
# Proper label mapping (id2label, label2id)
id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}

# AutoModelForTokenClassification using DistilBERT with correct num_labels
model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)
print("Model setup complete.\n")

# --- Setup for Task 5 Evaluation Metric ---
seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Remove ignored index (-100)
    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

# ==========================================
# TASK 4: Training

print("--- TASK 4: Training ---")
args = TrainingArguments(
    output_dir="./distilbert-finetuned-chunking",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

# Hugging Face Trainer implementation
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,     # --- CHANGED FROM tokenizer=tokenizer
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Starting training (this may take a few minutes)...")
trainer.train()

# ==========================================
# TASK 5: Evaluation 

print("--- TASK 5: Evaluation ---")

# We create a new arguments object with disable_tqdm=True to prevent the UI crash
eval_args = TrainingArguments(
    output_dir="./eval_output",
    disable_tqdm=True
)

# Create a fresh trainer just for evaluation using your already-trained model
eval_trainer = Trainer(
    model=model,
    args=eval_args,
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

eval_results = eval_trainer.evaluate()
print(f"Precision: {eval_results['eval_precision']:.4f}")
print(f"Recall: {eval_results['eval_recall']:.4f}")
print(f"F1 Score: {eval_results['eval_f1']:.4f}")

# ==========================================
# TASK 6: Inferenc

print("\n--- TASK 6: Inference ---")
token_classifier = pipeline(
    "token-classification",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple"
)

custom_sentence = "John works at Google in California"
predictions = token_classifier(custom_sentence)

print(f"Input: {custom_sentence}")
print("Output Chunk Tags:")
for entity in predictions:
    print(f"  {entity['word']:<12} -> {entity['entity_group']}")

### Task 7: Comparison (POS Tagging vs Chunking)
While both are sequence labeling tasks, they differ in complexity and scope:
* **POS Tagging (Easy):** This is grammar-level tagging. It assigns a part of speech (noun, verb, adjective) to every single, individual token in a sentence.
* **Chunking (Medium):** This is phrase-level grouping. It groups adjacent tokens into larger syntactic units, such as Noun Phrases (NP) or Verb Phrases (VP). Chunking often relies on the foundational data provided by POS tagging to make these groupings.

### Task 8: Report & Insights
**Challenges Faced:**
The primary challenge was managing subword tokenization. Because DistilBERT uses WordPiece tokenization, a single word might be split into multiple tokens. Aligning the original chunk tags to these new subwords without confusing the model required careful mapping—specifically, assigning the label to the first subword and masking the rest with the `-100` label so the loss function would ignore them.

**Observations:**
Evaluating token classification models requires specialized metrics. Standard accuracy is misleading because it doesn't account for the structure of multi-word entities. Using the `seqeval` library allowed for a true entity-level evaluation of Precision, Recall, and F1 Score. Additionally, the pre-trained DistilBERT model adapted incredibly fast, demonstrating the efficiency of transfer learning for NLP tasks.